# Esplorazione dei dati

Carica tutto quello che la pipeline ha scritto in `data/`. Prima di aprirlo, esegui gli script `01` → `05`.

In [1]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

import pandas as pd
import plotly.express as px
from src.data import read_weekly

pd.set_option("display.width", 200)
DATA = ROOT / "data"
print(ROOT)

/Users/antonellamosca/Desktop/model for investment/luiss-trading-strategy-skeleton


## 1. Prezzi settimanali (aggiustati per split e dividendi)

In [2]:
close = read_weekly(DATA / "prices" / "close.csv")
opens = read_weekly(DATA / "prices" / "open.csv")
close.tail()

,,AAPL,MSFT,AMZN,GOOG,SPY
date_open,date_close,,,,,
2026-08-24,2026-08-28,319.700012,513.530029,266.429993,342.657532,767.444275
2026-08-31,2026-09-04,319.970001,499.700012,258.510010,335.309998,768.282227
2026-09-07,2026-09-11,332.269989,495.630005,256.779999,335.450012,762.396790
2026-09-14,2026-09-18,336.130005,493.779999,253.710007,344.410004,761.690002
2026-09-21,2026-09-25,341.070007,516.169983,249.669998,341.079987,771.349976


In [3]:
close.describe().T.assign(missing=close.isna().sum(), first=close.apply(lambda s: s.first_valid_index()[1].date()))

,count,mean,std,min,25%,50%,75%,max,missing,first
AAPL,1395.0,54.969300,79.105078,0.196207,2.233502,16.506466,66.626450,341.070007,0,2000-01-07
MSFT,1395.0,107.864001,139.304651,11.163311,18.985903,27.796463,149.817657,519.337097,0,2000-01-07
AMZN,1395.0,55.743153,72.554128,0.298500,2.532000,13.643500,94.490749,274.480011,0,2000-01-07
GOOG,1154.0,64.616715,76.525456,2.467541,13.362500,31.195279,91.678402,396.553741,241,2004-08-20
SPY,1395.0,209.611099,172.857602,50.146885,84.624199,128.402374,272.449020,774.416992,0,2000-01-07


In [4]:
# Crescita di 100 investiti dalla prima data disponibile di ogni titolo (scala log)
norm = close.droplevel("date_open")
norm = 100 * norm / norm.bfill().iloc[0]
px.line(norm, log_y=True, title="Prezzi normalizzati a 100").show()

## 2. Rendimenti settimanali: controllo outlier

In [5]:
returns = close.droplevel("date_open").pct_change(fill_method=None)
print("Peggiori settimane:"); print(returns.min().round(3), returns.idxmin().dt.date, sep="\n")
returns.describe().T

Peggiori settimane:
AAPL   -0.507
MSFT   -0.190
AMZN   -0.320
GOOG   -0.154
SPY    -0.198
dtype: float64
AAPL    2000-09-29
MSFT    2000-12-01
AMZN    2000-12-22
GOOG    2008-11-21
SPY     2008-10-10
dtype: object


,count,mean,std,min,25%,50%,75%,max
AAPL,1394.0,0.005715,0.050137,-0.506587,-0.021256,0.006944,0.033488,0.266435
MSFT,1394.0,0.002691,0.038365,-0.190349,-0.017433,0.001868,0.022156,0.217501
AMZN,1394.0,0.005138,0.065320,-0.319672,-0.025245,0.002687,0.033914,0.752688
GOOG,1153.0,0.005096,0.042280,-0.153506,-0.019924,0.005350,0.027942,0.269367
SPY,1394.0,0.001837,0.024521,-0.197933,-0.009947,0.003000,0.014793,0.132923


In [6]:
px.histogram(returns.melt(var_name="ticker", value_name="ret").dropna(), x="ret", color="ticker",
             barmode="overlay", nbins=200, title="Distribuzione rendimenti settimanali").show()

## 3. Segnali delle strategie

In [7]:
signals = {p.stem: read_weekly(p) for p in sorted((DATA / "signals").glob("*.csv"))}
for name, s in signals.items():
    print(name, s.shape); display(s.tail(3))

random (1395, 4)


,,AAPL,MSFT,AMZN,GOOG
date_open,date_close,,,,
2026-09-07,2026-09-11,-1,-1,1,1.0
2026-09-14,2026-09-18,-1,1,1,0.0
2026-09-21,2026-09-25,0,1,0,0.0


sma (1386, 4)


,,AAPL,MSFT,AMZN,GOOG
date_open,date_close,,,,
2026-09-07,2026-09-11,1,1,0,0.0
2026-09-14,2026-09-18,1,1,0,1.0
2026-09-21,2026-09-25,1,1,0,0.0


## 4. Risultati del backtest

In [8]:
for name in ["performance", "return_stats", "behaviour", "market"]:
    print(f"--- {name} ---"); display(pd.read_csv(DATA / "metrics" / f"{name}.csv"))

--- performance ---


,period,series,cumulative_return,annualized_return,annualized_volatility,sharpe_ratio,sortino_ratio,max_drawdown,calmar_ratio
0,IS,sma,7.557035,0.163063,0.154388,0.926649,1.532729,-0.223672,0.729028
1,IS,random,0.653980,0.036041,0.128695,0.124644,0.190092,-0.243848,0.147801
2,IS,Equal weight,24.217938,0.254966,0.237766,0.988221,1.563643,-0.537657,0.474216
3,IS,SPY,1.909796,0.078052,0.169300,0.342896,0.478800,-0.552346,0.141311
4,OOS,sma,1.558053,0.128504,0.147421,0.736015,1.140675,-0.245890,0.522609
5,OOS,random,-0.101993,-0.013751,0.122148,-0.276315,-0.386294,-0.285189,-0.048218
6,OOS,Equal weight,5.886983,0.281932,0.250480,1.045720,1.648466,-0.368786,0.764488
7,OOS,SPY,2.592179,0.178913,0.181535,0.875385,1.298313,-0.317905,0.562787


--- return_stats ---


,period,series,hit_rate,avg_gain,avg_loss,profit_factor,best,worst,skew,kurtosis
0,IS,sma,0.499323,0.018005,-0.014917,1.535796,0.132423,-0.096819,0.354517,3.995549
1,IS,random,0.499323,0.013576,-0.012091,1.141355,0.087691,-0.065820,0.319588,2.374647
2,IS,Equal weight,0.564276,0.026162,-0.022591,1.499722,0.132423,-0.141910,-0.038746,2.194435
3,IS,SPY,0.576455,0.015540,-0.017080,1.238316,0.132597,-0.123939,-0.527109,5.795386
4,OOS,sma,0.490099,0.017470,-0.016562,1.420819,0.061566,-0.086071,-0.101995,1.527121
5,OOS,random,0.487624,0.012482,-0.012297,0.980207,0.064546,-0.060397,-0.086654,1.708833
6,OOS,Equal weight,0.596535,0.026741,-0.026183,1.510070,0.149794,-0.113631,-0.043755,1.425871
7,OOS,SPY,0.621287,0.017245,-0.019081,1.482633,0.120557,-0.123937,-0.320739,5.092384


--- behaviour ---


,period,series,net_exposure,gross_exposure,long,flat,short,turnover_per_year,avg_holding
0,IS,sma,0.635995,0.635995,0.635995,0.364005,0.000000,8.092016,8.173913
1,IS,random,-0.001691,0.666103,0.332206,0.333897,0.333897,46.405954,1.491667
2,OOS,sma,0.637995,0.637995,0.637995,0.362005,0.000000,8.945545,7.364286
3,OOS,random,-0.012995,0.671411,0.329208,0.328589,0.342203,46.980198,1.482240


--- market ---


,period,series,beta,correlation,alpha,tracking_error,information_ratio,down_market,up_market,sharpe_vs_market
0,IS,sma,0.461719,0.506318,0.110941,0.161338,0.455258,-0.007652,0.011063,0.583753
1,IS,random,-0.049175,-0.064690,0.027082,0.219189,-0.209824,0.001937,0.000033,-0.218251
2,IS,Equal weight,1.033382,0.735815,0.163817,0.161110,1.031233,-0.015598,0.019994,0.645325
3,OOS,sma,0.403271,0.496589,0.046798,0.167656,-0.295188,-0.011284,0.010960,-0.139370
4,OOS,random,-0.001125,-0.001673,-0.026209,0.218973,-0.857414,-0.000763,0.000267,-1.151700
5,OOS,Equal weight,1.122219,0.813325,0.079110,0.147411,0.670446,-0.022625,0.022464,0.170336


In [9]:
ret = read_weekly(DATA / "metrics" / "returns.csv")
ret.head()

,,period,sma,random,Equal weight,SPY
date_open,date_close,,,,,
2004-10-18,2004-10-22,IS,0.056805,0.056805,0.058422,0.034715
2004-10-25,2004-10-29,IS,-0.011733,0.019110,0.006750,0.030116
2004-11-01,2004-11-05,IS,0.029273,-0.007030,0.054511,0.015996
2004-11-08,2004-11-12,IS,-0.010124,-0.046635,-0.010124,-0.011223
2004-11-15,2004-11-19,IS,0.052049,-0.031698,0.059594,0.013826
